# Face portrait prompt generation

Generate portrait prompts from attribute distributions using Qwen3-14B. Configure paths and settings in the next cell. Requires a CUDA GPU and the dependency packages listed in `requirements.txt`.

In [ ]:
from pathlib import Path
import math
import random
import re

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Paths (relative to the notebook's working directory, or use absolute paths).
ATTRIBUTES_DIR = Path("attributes")
OUTPUT_DIR = Path("prompts_files")

# Sampling and model settings.
NUM_IDENTITIES = 8000  
MULTIPLIER = 4000  # Per skin condition
RANDOM_SEED = 42
GENDER_COLUMN = "genders"
MODEL_NAME = "Qwen/Qwen3-14B"
MAX_NEW_TOKENS = 220

# Required conditional attribute file stems (.txt or .csv).
SCALP_HAIR_MALE_FILE = "scalpHair_male"
SCALP_HAIR_FEMALE_FILE = "scalpHair_female"
FACIAL_HAIR_FILE = "facialHair"
TATTOO_FILE = "tattoo"
FACE_AREAS_FILE = "face_area"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_probability_file(file_path):
    rows = []
    with open(file_path, "r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line or line.startswith("```") or "," not in line:
                continue
            value, probability = line.rsplit(",", 1)
            probability = float(probability.strip())
            if probability > 0:
                rows.append((value.strip(), probability))

    if not rows:
        raise ValueError(f"No valid probability entries found in {file_path}")
    total = sum(probability for _, probability in rows)
    return [(value, probability / total) for value, probability in rows]


def find_attribute_file(attributes_dir, stem):
    for extension in (".txt", ".csv"):
        path = attributes_dir / f"{stem}{extension}"
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected {stem}.txt or {stem}.csv in {attributes_dir}")


def load_all_attribute_files(attributes_dir):
    excluded = {
        SCALP_HAIR_MALE_FILE, SCALP_HAIR_FEMALE_FILE,
        FACIAL_HAIR_FILE, TATTOO_FILE, FACE_AREAS_FILE,
    }
    paths = sorted((*attributes_dir.glob("*.txt"), *attributes_dir.glob("*.csv")))
    if not paths:
        raise FileNotFoundError(f"No attribute files in {attributes_dir}")

    attributes = {}
    for path in paths:
        if path.stem not in excluded:
            if path.stem in attributes:
                raise ValueError(f"Duplicate attribute stem: {path.stem}")
            attributes[path.stem] = read_probability_file(path)

    if GENDER_COLUMN not in attributes:
        raise ValueError(f"Missing gender attribute: {GENDER_COLUMN}.txt or .csv")
    return attributes


attributes = load_all_attribute_files(ATTRIBUTES_DIR)
conditional_attributes = {
    "scalpHair": {
        "male": read_probability_file(find_attribute_file(ATTRIBUTES_DIR, SCALP_HAIR_MALE_FILE)),
        "female": read_probability_file(find_attribute_file(ATTRIBUTES_DIR, SCALP_HAIR_FEMALE_FILE)),
    },
    "facialHair": {
        "male": read_probability_file(find_attribute_file(ATTRIBUTES_DIR, FACIAL_HAIR_FILE)),
        "female": [("None", 1.0)],
        "non-binary": [("None", 1.0)],
    },
}
tattoo_attributes = read_probability_file(find_attribute_file(ATTRIBUTES_DIR, TATTOO_FILE))
face_area_attributes = read_probability_file(find_attribute_file(ATTRIBUTES_DIR, FACE_AREAS_FILE))

In [ ]:
def balanced_sample_from_distribution(items, n, seed=None):
    rng = random.Random(seed)
    values = [item[0] for item in items]
    expected = [probability * n for _, probability in items]
    counts = [math.floor(value) for value in expected]
    remaining = n - sum(counts)
    ranked = sorted(range(len(values)), key=lambda i: expected[i] - counts[i], reverse=True)
    for i in ranked[:remaining]:
        counts[i] += 1
    sampled = [value for value, count in zip(values, counts) for _ in range(count)]
    rng.shuffle(sampled)
    return sampled


def normalize_gender_value(gender):
    value = str(gender).strip().lower().replace("_", "-").replace(" ", "-")
    if value in ("male", "man", "masculine"):
        return "male"
    if value in ("female", "woman", "feminine"):
        return "female"
    if value in ("non-binary", "nonbinary", "non-binary-person", "nb"):
        return "non-binary"
    return value


def sample_one(distribution, rng):
    values, probabilities = zip(*distribution)
    return rng.choices(values, weights=probabilities, k=1)[0]


def quota_based_skin_conditions(distribution, multiplier, seed=None):
    sampled = []
    for value, _ in distribution:
        count = multiplier * (2 if str(value).strip().lower() == "natural-face" else 1)
        sampled.extend([value] * count)
    random.Random(seed).shuffle(sampled)
    return sampled


def generate_attribute_sets(attributes, conditional_attributes, multiplier, seed=42):
    rng = random.Random(seed)
    if "skin_condition" in attributes:
        skin_conditions = quota_based_skin_conditions(attributes["skin_condition"], multiplier, seed)
        n = len(skin_conditions)
    else:
        skin_conditions = None
        n = NUM_IDENTITIES

    sampled_columns = {}
    if skin_conditions is not None:
        sampled_columns["skin_condition"] = skin_conditions
    for index, (name, distribution) in enumerate(attributes.items()):
        if name != "skin_condition":
            sampled_columns[name] = balanced_sample_from_distribution(distribution, n, seed + index)

    rows = []
    for i in range(n):
        row = {name: sampled_columns[name][i] for name in attributes}
        if skin_conditions is not None:
            condition = str(row["skin_condition"]).strip().lower()
            if condition == "tattoo":
                area = sample_one(face_area_attributes, rng)
                design = sample_one(tattoo_attributes, rng)
                row["skin_condition"] = f"tattoo ({design}) on {area}"
            elif condition != "natural-face":
                area = sample_one(face_area_attributes, rng)
                row["skin_condition"] = f"{row['skin_condition']} on {area}"

        gender = normalize_gender_value(row[GENDER_COLUMN])
        scalp_group = gender if gender in ("male", "female") else rng.choice(("male", "female"))
        row["scalpHair"] = sample_one(conditional_attributes["scalpHair"][scalp_group], rng)
        row["facialHair"] = sample_one(conditional_attributes["facialHair"].get(gender, [("None", 1.0)]), rng)
        rows.append(row)
    rng.shuffle(rows)
    return rows


attribute_sets = generate_attribute_sets(attributes, conditional_attributes, MULTIPLIER, RANDOM_SEED)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA-enabled GPU for 4-bit model inference.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
pad_token_id = tokenizer.pad_token_id
eos_token_id = tokenizer.eos_token_id
if isinstance(eos_token_id, list):
    eos_token_id = eos_token_id[0]
model.generation_config.pad_token_id = pad_token_id
model.generation_config.eos_token_id = eos_token_id

In [ ]:
def attributes_to_text(attribute_dict):
    return "\n".join(
        f"{key.replace('_', ' ').replace('-', ' ')}: {str(value).replace('-', ' ')}"
        for key, value in attribute_dict.items()
    )


def clean_generated_prompt(text):
    text = re.sub(r"<think>.*?</think>", "", text.strip(), flags=re.DOTALL).strip()
    for marker in ("FINAL_PROMPT:", "Final prompt:", "Prompt:", "Image prompt:"):
        if marker in text:
            text = text.split(marker, 1)[-1].strip()
    ignored = ("1.", "Thinking Process", "**Analyze", "Analyze the Request", "Here is")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    text = " ".join(line for line in lines if not line.startswith(ignored)
                    and not any(tag in line for tag in ("Analyze the Request", "Goal:", "Constraints:")))
    return text.strip().strip('"').strip("'").strip()


def generate_prompt_with_qwen(attribute_dict):
    attribute_text = attributes_to_text(attribute_dict)
    system_message = (
        "You are an expert prompt writer for realistic portrait photography. "
        "Create concise, natural, visually specific prompts in English language for generating face portraits with given attributes. "
        "The final image should look like a real photograph of an ordinary person, not AI-generated. "
        "Do not use the words synthetic, artificial, generated, CGI, render, 3D, doll-like, flawless, or perfect. "
        "Do not mention probability values. "
        "Do not add named celebrities, copyrighted characters, public figures, or real people. "
        "Do not add identity attributes that conflict with the provided attributes. "
        "Avoid stereotypes and avoid exaggerated beauty language. "
        "Return only the final prompt text."
    )

    user_message = f"""
Create one realistic face portrait prompt using these attributes:

{attribute_text}

Requirements:
- Write as a natural photographic portrait, not as an AI/synthetic image where attributes should be described in a detailed way, making them look clearly visible and realistic, not just subtle.
- Write a concise prompt while mentioning all the provided attributes.
- Use all the provided attributes in a cohesive way to describe the person's appearance and style.
- Describe the person's face, attributes, skin texture, hair, age impression, and other provided attributes.
- Include natural imperfections: subtle pores, slight facial asymmetry, fine skin texture, small flyaway hairs, and realistic eyes.
- Use believable lighting and camera language: natural window light, soft shadows, shallow depth of field, close-up portrait, 85mm lens, realistic color tones.
- Keep the person neutral and ordinary-looking, not overly idealized.
- Do not use words like perfect, flawless, symmetrical, beautiful, model, synthetic, AI-generated, CGI, render, plastic, waxy, or doll-like.
- Do not include explanations.
- Do not use foreign languages, keep the prompt in English.
- Return only one prompt.
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    try:
        chat_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(chat_text, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=40,
            repetition_penalty=1.1,
            pad_token_id=pad_token_id,
            eos_token_id=eos_token_id,
            use_cache=True,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return clean_generated_prompt(tokenizer.decode(generated_ids, skip_special_tokens=True))

In [ ]:
def safe_filename(value):
    return re.sub(r"[^a-z0-9_]+", "", str(value).strip().lower().replace(" ", "_").replace("-", "_"))


def save_csvs_by_gender(df, output_dir, gender_column):
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_files = {}
    for gender, group in df.groupby(gender_column):
        key = safe_filename(gender)
        if not key:
            raise ValueError(f"Invalid gender value for filename: {gender!r}")
        path = output_dir / f"face_generation_{key}.csv"
        group.to_csv(path, index=False, encoding="utf-8")
        saved_files[key] = str(path)
    return saved_files


results = []
for index, attribute_dict in enumerate(tqdm(attribute_sets, desc="Generating prompts"), start=1):
    row = {"prompt_id": index, "prompt": generate_prompt_with_qwen(attribute_dict)}
    row.update(attribute_dict)
    results.append(row)

df_prompts = pd.DataFrame(results)
saved_files = save_csvs_by_gender(df_prompts, OUTPUT_DIR, GENDER_COLUMN)
print(f"Saved {len(df_prompts)} prompts to {len(saved_files)} file(s) in {OUTPUT_DIR}")